# Continuous-Time Markov Chain Exploration

This notebook runs the CTMC progression for the capstone project:

1. Global CTMC baseline
2. Clustered CTMC segmentation
3. Personalized neural-style CTMC rates
4. Benchmark model comparison

The code uses checkpointed parquet files from the local `data/` folder.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src" / "models"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "visualizations"))
sys.path.insert(0, str(PROJECT_ROOT / ".codex_deps"))

import matplotlib.pyplot as plt
import pandas as pd

from ctmc import (
    CTMCData,
    ClusteredCTMC,
    GlobalCTMC,
    ModelComparison,
    NeuralRateCTMC,
    sample_pipeline,
)
from ctmc_submission import create_ctmc_submissions
from ctmc_plots import (
    plot_absorption_by_state,
    plot_calibration,
    plot_generator_heatmap,
    plot_metric_comparison,
    plot_top_transition_graph,
)
from tabular_submission import create_tabular_submissions

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Increase this once the workflow is settled.
MAX_JOURNEYS = 50_000
N_CLUSTERS = 4

## Load Transition Data

Each row below is one observed transition: current action, next action, and elapsed time.

In [ ]:
data = CTMCData()
transitions = data.transition_table(max_journeys=MAX_JOURNEYS)

print(transitions.shape)
transitions.head()

## 1. Baseline Global CTMC

Estimate transition counts $N_{ij}$, time in state $T_i$, and global generator rates $q_{ij}=N_{ij}/T_i$.

In [ ]:
global_ctmc = GlobalCTMC().fit(transitions)

print("Q shape:", global_ctmc.Q_.shape)
display(global_ctmc.transition_counts_.head())
display(global_ctmc.time_in_state_.sort_values(ascending=False).head(10).rename("seconds_in_state"))
display(global_ctmc.top_rates(20))

plot_generator_heatmap(global_ctmc, RESULTS_DIR / "ctmc_generator_heatmap.png")
plot_top_transition_graph(global_ctmc, RESULTS_DIR / "ctmc_top_transition_graph.png", n_edges=25)
absorption_by_state = plot_absorption_by_state(global_ctmc, RESULTS_DIR / "ctmc_absorption_by_state.png")
absorption_by_state.to_csv(RESULTS_DIR / "ctmc_absorption_by_state.csv", index=False)
display(absorption_by_state)

In [ ]:
q_abs = global_ctmc.Q_.copy()
for state in q_abs.index:
    q_abs.loc[state, state] = 0

plt.figure(figsize=(10, 8))
plt.imshow(q_abs, aspect="auto")
plt.colorbar(label="rate")
plt.title("Global CTMC off-diagonal transition rates")
plt.xlabel("to state index")
plt.ylabel("from state index")
plt.tight_layout()
plt.show()

In [ ]:
# P(t) = exp(Qt): probability of being in each state after a horizon.
one_day = 24 * 60 * 60
p_1day = global_ctmc.transition_probability(one_day)
p_1day.head()

## 2. Segmentation: Clustered CTMC

Cluster users using action counts/proportions, observed time in states, time to key actions, and prefix metadata. The clusterer does not use the purchase label.

In [ ]:
clustered = ClusteredCTMC(n_clusters=N_CLUSTERS, random_state=42).fit(transitions)
cluster_summary = clustered.cluster_summary()
display(cluster_summary)

cluster_summary.plot.bar(x="cluster", y="n_journeys", legend=False, figsize=(6, 4))
plt.title("Journey clusters")
plt.ylabel("n journeys")
plt.tight_layout()
plt.show()

In [ ]:
for cluster_id, model in clustered.models_.items():
    print(f"Cluster {cluster_id}")
    display(model.top_rates(10))

## 3. Neural Exponential-Rate CTMC

The neural rate model learns a customer-dependent success intensity, without predicting the next state:

$$P(T_{success} \le t \mid x) = 1 - \exp(-\lambda(x)t).$$

This is simpler than a full neural CTMC, but it directly estimates the exponential waiting-time rate relevant to the 60-day success horizon.

In [ ]:
features = clustered.feature_builder.transform(transitions)
display(features.head())

neural_training = data.load_neural_rate_training_features(max_rows=100_000)
neural_ctmc = NeuralRateCTMC(hidden_layer_sizes=(64, 32), random_state=42)
neural_ctmc.fit(neural_training)

example_rows = neural_training.head(10)
pd.DataFrame({
    "id": example_rows["id"],
    "label": example_rows["label"],
    "lambda_hat": neural_ctmc.predict_lambda(example_rows),
    "p_success_60d": neural_ctmc.predict_success_probability(example_rows),
})

## 4. Model Comparison

Compare against standard predictive baselines on the truncated training features created by the pipeline.

In [ ]:
training_df = data.load_training_features(max_rows=100_000)
print(training_df.shape)
training_df.head()

In [ ]:
comparison = ModelComparison(random_state=42).run(training_df)
comparison.to_csv(RESULTS_DIR / "tabular_baseline_comparison.csv", index=False)
display(comparison)

comparison.plot.bar(x="model", y=["roc_auc", "average_precision"], figsize=(8, 4))
plt.ylim(0, 1)
plt.title("Benchmark comparison")
plt.tight_layout()
plt.show()

comparison.plot.bar(x="model", y=["log_loss", "brier_score"], figsize=(8, 4))
plt.title("Benchmark probability loss")
plt.tight_layout()
plt.show()

## 5. CTMC vs Baselines

This cell evaluates the three CTMC variants on journeys in the sampled transition table whose labels are available in the engineered training data. It is an exploration metric, not the official Kaggle score.

**Leakage warning:** if CTMC features are built from full completed journeys, successful journeys can expose final state `28` (`order_shipped`). Treat CTMC scores here as diagnostic unless the evaluation uses the same truncated/open-journey observation window as Kaggle.

In [ ]:
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score

label_lookup = data.load_binary_labels()
ctmc_eval = features.merge(label_lookup, on="id", how="inner")
ctmc_eval["state"] = ctmc_eval["current_state"]

ctmc_rows = []
ctmc_predictions = {
    "ctmc_global": global_ctmc.absorption_probability(ctmc_eval["current_state"]),
    "ctmc_clustered": clustered.predict_success_probability(ctmc_eval, fallback_model=global_ctmc),
    "ctmc_neural_rate": neural_ctmc.predict_success_probability(ctmc_eval),
}

for name, probs in ctmc_predictions.items():
    ctmc_rows.append({
        "model": name,
        "roc_auc": roc_auc_score(ctmc_eval["label"], probs),
        "average_precision": average_precision_score(ctmc_eval["label"], probs),
        "log_loss": log_loss(ctmc_eval["label"], probs, labels=[0, 1]),
        "brier_score": brier_score_loss(ctmc_eval["label"], probs),
    })

ctmc_comparison = pd.concat([comparison, pd.DataFrame(ctmc_rows)], ignore_index=True)
ctmc_comparison = ctmc_comparison.sort_values("roc_auc", ascending=False)
ctmc_comparison.to_csv(RESULTS_DIR / "ctmc_vs_baselines_comparison.csv", index=False)
plot_metric_comparison(ctmc_comparison, RESULTS_DIR / "ctmc_vs_baselines_metrics.png")
display(ctmc_comparison)

ctmc_comparison.plot.bar(x="model", y=["roc_auc", "average_precision"], figsize=(10, 4))
plt.ylim(0, 1)
plt.title("CTMC variants vs tabular baselines")
plt.tight_layout()
plt.show()

ctmc_comparison.sort_values("log_loss").plot.bar(x="model", y=["log_loss", "brier_score"], figsize=(10, 4))
plt.title("CTMC variants vs baselines: probability loss")
plt.tight_layout()
plt.show()

for name, probs in ctmc_predictions.items():
    calibration = plot_calibration(
        ctmc_eval["label"],
        probs,
        RESULTS_DIR / f"{name}_calibration.png",
        title=f"{name} calibration",
    )
    calibration.to_csv(RESULTS_DIR / f"{name}_calibration.csv", index=False)

## 6. Write Kaggle Submission CSVs

This writes the model outputs to `results/`. The cell expects `data/open_journeys1.csv` to exist. If `data/open_journeys1_flattened_all0.csv` is missing, the submission helper creates the all-zero ID template from the open journeys file.

In [ ]:
TEST_EVENTS = PROJECT_ROOT / "data" / "open_journeys1.csv"
SAMPLE_TEMPLATE = PROJECT_ROOT / "data" / "open_journeys1_flattened_all0.csv"

if not TEST_EVENTS.exists():
    print(f"Missing {TEST_EVENTS}. Add the Kaggle open journey event file before writing submission CSVs.")
else:
    ctmc_outputs = create_ctmc_submissions(
        test_events_path=TEST_EVENTS,
        sample_path=SAMPLE_TEMPLATE,
        output_dir=RESULTS_DIR,
        max_train_journeys=MAX_JOURNEYS,
        n_clusters=N_CLUSTERS,
        neural_transition_limit=75_000,
    )
    tabular_outputs = create_tabular_submissions(
        test_events_path=TEST_EVENTS,
        sample_path=SAMPLE_TEMPLATE,
        output_dir=RESULTS_DIR,
        max_train_rows=300_000,
    )
    print("Wrote submission files:")
    for path in sorted(RESULTS_DIR.glob("*_submission.csv")):
        print(path.name)


## Decision Notes

- Global CTMC: most interpretable and gives direct time-to-event dynamics.
- Clustered CTMC: keeps interpretability while capturing segment-level behavior.
- Personalized rates: more flexible, but less transparent and more tuning-sensitive.
- Tree/boosting baselines: likely stronger for raw prediction on engineered tabular features.
- Transformer/sequence model: likely best if the team has time to build a proper sequence validation setup.